# Попробовать распознаватель на картинках строк

Укажи чекпойнт и строки (папку с картинками, один файл или список файлов) и выполни ячейки по порядку.
Каждая строка читается обеими головами: `ctc` и `decoder`. Число рядом с текстом — уверенность от 0 до 1.

In [ ]:
import sys
import time
from pathlib import Path

from IPython.display import display
from PIL import Image

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "htr").is_dir())
sys.path.insert(0, str(ROOT))
from htr.predictor import Recognizer

CHECKPOINT = ROOT / "outputs/finetune/best.pt"
LINES = Path("/path/to/lines")     # папка, файл или список файлов
DEVICE = "cuda"                    # или "cpu"
LIMIT = 50                         # сколько строк взять из папки

In [ ]:
reader = Recognizer(CHECKPOINT, DEVICE)
size = sum(p.numel() for p in reader.model.parameters()) / 1e6
print(f"{size:.1f}M параметров, {len(reader.alphabet.chars)} символов, высота строки {reader.height} px")

if isinstance(LINES, Path) and LINES.is_dir():
    kinds = {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff"}
    paths = sorted(p for p in LINES.iterdir() if p.suffix.lower() in kinds)[:LIMIT]
else:
    paths = [Path(p) for p in ([LINES] if isinstance(LINES, (str, Path)) else LINES)]
images = [Image.open(p).convert("RGB") for p in paths]
print(len(images), "строк")

In [ ]:
def read(head):
    reader.head = head
    return reader.read(images)


for path, image, ctc, decoder in zip(paths, images, read("ctc"), read("decoder")):
    shown = image.copy()
    shown.thumbnail((1100, 130))
    display(shown)
    print(f"{path.name}\n  ctc      {ctc[1]:.2f}  {ctc[0]}\n  decoder  {decoder[1]:.2f}  {decoder[0]}\n")

## Скорость

`ctc` — один проход без цикла, `decoder` — буква за буквой, `auto` — декодер только там, где CTC не уверена.

In [ ]:
for head in ("ctc", "decoder", "auto"):
    reader.head = head
    start = time.perf_counter()
    reader.recognize(images)
    print(f"{head:8} {1000 * (time.perf_counter() - start) / len(images):.1f} мс на строку")

## Строки со страницы

Модель училась на строках с небольшим запасом бумаги вокруг текста. Если строки берутся со страницы по боксам детектора, резать их нужно той же функцией, что при обучении:

```python
from htr.geometry import crop_line, paper_colour

page = Image.open("page.jpg").convert("RGB")
crops = [crop_line(page, box, reader.frame, paper_colour(page)) for box in boxes]   # boxes: четырёхугольники детектора
texts = reader.recognize(crops)
```